In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["KERAS_BACKEND"] = "torch"

import src.util.supabase_utils as su
import src.config as config
import src.data_preprocessing.text_preprocessing as text_pre
from src.data_preprocessing.TextEmbedder import TextEmbedder
from src.agents.util.Retriever import Retriever
from src.agents.util.ResponseModel import ResponseModel
from src.agents.NLP_Agent import NLPAgent
from src.agents.Visual_Agent import VisualModel
import src.data_preprocessing.image_preprocessing as img_pre
import keras
from src.data_preprocessing.build_training_sets import load_vqa_dataset

In [ ]:
plant_expert_vqa_TRAIN, plant_expert_vqa_TEST, plant_expert_vqa_VAL = load_vqa_dataset()

In [ ]:
embedder = TextEmbedder("sentence-transformers/all-MiniLM-L6-v2")
client = su.connect_supabase()
retriever = Retriever(client, embedder)

retriever.retrieve("how do I treat leaf spot")[:1]

In [ ]:
train_sample = (plant_expert_vqa_TRAIN.drop_duplicates(subset=["question_text", "answer"]).sample(5000, random_state=0))

In [ ]:
model = ResponseModel(lora=True)

In [ ]:
model.train_model(train_sample, retriever=None, epochs=1,
                  out="./models/response_models/10_10")

In [ ]:

retriever = Retriever(client, embedder, match_threshold=.5)
unet = keras.models.load_model("models/unet_models/UNet_Best.keras", compile=False)
visual_model = VisualModel.load("models/visual_models/10_10.keras",
                                "models/visual_models/10_10_classes.json", unet=unet)
response_model = ResponseModel(adapter="models/response_models/10_9")
agent = NLPAgent(retriever=retriever, response_model=response_model, visual_model=visual_model)

for row in plant_expert_vqa_VAL.sample(3, random_state=0).itertuples():
    image = img_pre.load_image(row.image_path)
    pred = visual_model.predict(image)
    visual_observations = pred.get_observations()

    answer = agent.ask_question(image=image, question=row.question_text)
    chunks = retriever.retrieve(
        f"{row.question_text} {visual_observations}",
        plant_code_filter=pred.get_plant_code(),
        genus_filter=pred.get_genus(),
        num_results=5
    )
    print("\n\n-----------QUESTION AND VISUAL ASSESSMENT------------\n\n")
    print(f"question: {row.question_text}")
    print(f"truth: crop={row.crop} | disease={row.disease} | severity={row.severity}")
    print(f"plant code: {pred.get_plant_code()}  genus: {pred.get_genus()}")
    for i, chunk in enumerate(chunks):
        print(f"\n\n----------ASSESSING CHUNK {i}----------\n\n")
        similarity = chunk.get("similarity")
        sim = f"{similarity:.3f}" if similarity is not None else "NONE"
        print(f"[{i}] similarity: {similarity}  {chunk.get('plant_code')}  {chunk.get('common_name')}  ({chunk.get('source_file')})")
        print(f"{chunk['body'][:300]}")
    print(f"\n\n----------ANSWER ASSESSMENT----------\n\n")
    print(f"true answer: {row.answer[:400]}")
    print(f"visual observations:  {visual_observations}")
    print(f"pred_answer: {answer[:400]}\n")

In [ ]:
from pathlib import Path
#load in the GBIF data
#GET THE PATH TO ROOT OF PROJECT
import pandas as pd

ROOT_PATH = Path.cwd()
print(ROOT_PATH)
TRAIN_DATA = pd.read_csv(config.Data.GBIF.TRAIN_FILE)
TEST_DATA = pd.read_csv(config.Data.GBIF.TEST_FILE)
VAL_DATA = pd.read_csv(config.Data.GBIF.VALIDATION_FILE)
gbif_row = TEST_DATA.sample(1, random_state=0).iloc[0]
image = img_pre.load_image(str(config.General.PROJECT_ROOT / gbif_row["image_path"].lstrip("/")))
pred = visual_model.predict(image)
print("truth:", gbif_row["symbol"], gbif_row["genus"])
print("pred: ", pred.get_plant_code(), pred.get_genus())
print(pred.classification_heads["symbol"][:3])